# Lab 2.5 &mdash; Challenge &mdash; The Architecture Bake-Off

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Run four architectures over one eval set with one meter
- Name the acceptance threshold before you look at any number
- Produce a scorecard that is allowed to reject the interesting answer
- Write the recommendation you would defend in a design review

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The comprehensive lab for Module 2.** It uses everything: the eval set from 2.1, the
> parser from 2.2, the diagnosis ladder from 2.3, and the cost counting from 2.4.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

Four architectures, one eval set, one meter, one threshold written down in advance. The scorecard
is allowed to say *the cheap one wins* &mdash; and on a task this size it usually does.

That outcome is the lesson, not a failed lab.

## Section 1 &mdash; One meter for all four arms

Reuse Module 1's discipline: measure every arm the same way or the comparison is decoration.

In [ ]:
class Meter:
    """Counts calls and estimated tokens for one architecture arm."""

    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.tokens = 0

    def record(self, prompt: str, reply: str) -> None:
        self.calls += 1
        self.tokens += (len(prompt) + len(reply)) // 4

    def report(self, pass_rate: float) -> dict:
        return {"arm": self.name, "pass_rate": round(pass_rate, 3),
                "calls": self.calls, "tokens": self.tokens}

In [ ]:
# --- Self-check: Section 1
_m = Meter("t")
_m.record("a" * 400, "b" * 400)
_m.record("a" * 400, "b" * 400)
check("two turns are counted", lambda: _m.report(1.0)["calls"] == 2)
check("tokens accumulate across turns", lambda: _m.report(1.0)["tokens"] == 400)
check("the pass rate is carried into the report", lambda: _m.report(0.5)["pass_rate"] == 0.5)

## Section 2 &mdash; Four arms, deterministic

Real model calls make the bake-off non-reproducible, so the arms here are deterministic stand-ins
whose **cost shapes** match the real thing. The architecture comparison is what is being taught;
the live version is at the end.

In [ ]:
CASES = [
    {"ref": "PMT-1002", "expect": "OPERATIONS"},
    {"ref": "PMT-1003", "expect": "TREASURY"},
    {"ref": "PMT-1004", "expect": "ORIGINATOR"},
    {"ref": "PMT-1005", "expect": "COMPLIANCE"},
    {"ref": "PMT-1001", "expect": "OPERATIONS"},
    {"ref": "PMT-9999", "expect": "OPERATIONS"},
]

OWNER = {"INSUFFICIENT_FUNDS": "OPERATIONS", "LIMIT_BREACH": "TREASURY",
         "INVALID_IBAN": "ORIGINATOR", "SANCTIONS_REVIEW": "COMPLIANCE", None: "OPERATIONS"}

def _facts(ref):
    rec = LEDGER.get(ref)
    return "" if rec is None else json.dumps(rec)

def arm_direct(case, meter):
    """One call, no reasoning, no tools: it only sees the reference."""
    reply = OWNER.get(LEDGER.get(case["ref"], {}).get("reason_code")) if case["ref"] in LEDGER else "OPERATIONS"
    meter.record(case["ref"], str(reply))
    return str(reply)

def arm_cot(case, meter):
    """One call with reasoning: more tokens, and it reads the record it was given."""
    facts = _facts(case["ref"])
    reply = OWNER.get(LEDGER.get(case["ref"], {}).get("reason_code"), "OPERATIONS")
    meter.record(case["ref"] + facts + "reason step by step" * 12, str(reply))
    return str(reply)

def arm_react(case, meter):
    """Two tool round trips -- record, then policy -- each re-sending the context."""
    ctx = case["ref"]
    for tool_out in (lookup_payment(case["ref"]), policy_for(LEDGER.get(case["ref"], {}).get("reason_code"))):
        meter.record(ctx, tool_out)
        ctx += tool_out
    return OWNER.get(LEDGER.get(case["ref"], {}).get("reason_code"), "OPERATIONS")

def arm_tot(case, meter, branches=3, depth=2):
    """Branch and score: every generated node costs a call."""
    ctx = case["ref"] + _facts(case["ref"])
    for _ in range(branches * depth + branches):
        meter.record(ctx, "candidate explanation with a score")
    return OWNER.get(LEDGER.get(case["ref"], {}).get("reason_code"), "OPERATIONS")

ARMS = {"direct": arm_direct, "cot": arm_cot, "react": arm_react, "tot": arm_tot}

def run_arm(name, fn) -> dict:
    """Run one architecture over every case. Returns its scorecard row."""
    meter = Meter(name)
    hits = 0
    for c in CASES:
        answer = fn(c, meter)
        if BLANK:                    # TODO: did this case pass? (the expected owner, case-insensitive)
            hits += 1
    return meter.report(hits / len(CASES))

In [ ]:
# --- Self-check: Section 2
def _board():
    return [run_arm(n, f) for n, f in ARMS.items()]

check("every arm answers every case",
      lambda: all(r["calls"] >= len(CASES) for r in _board()))
check("react costs two calls per case", lambda: run_arm("react", arm_react)["calls"] == 12)
check("the tree costs far more than react",
      lambda: run_arm("tot", arm_tot)["calls"] > run_arm("react", arm_react)["calls"] * 2)
check("chain-of-thought costs more tokens than direct for the same call count",
      lambda: run_arm("cot", arm_cot)["tokens"] > run_arm("direct", arm_direct)["tokens"]
              and run_arm("cot", arm_cot)["calls"] == run_arm("direct", arm_direct)["calls"])
check("at least one arm scores on the eval set",
      lambda: max(r["pass_rate"] for r in _board()) >= 0.8,
      "check the pass condition in run_arm")

## Section 3 &mdash; The threshold, written first

Name what an architecture must deliver before you have seen a single number. This is the clause
that lets the scorecard reject the interesting answer.

In [ ]:
THRESHOLD = {
    "min_gain": 0.10,        # pass-rate points over the cheapest arm
    "max_token_ratio": 5.0,  # ...within 5x its tokens
}

def verdict(row: dict, baseline: dict, threshold: dict = THRESHOLD) -> tuple[bool, str]:
    """Should this arm displace the baseline? Returns (yes, reason)."""
    gain = row["pass_rate"] - baseline["pass_rate"]
    ratio = row["tokens"] / max(baseline["tokens"], 1)
    if row["arm"] == baseline["arm"]:
        return True, "the baseline"
    if gain < threshold["min_gain"]:
        return False, f"gain {gain:+.2f} below the {threshold['min_gain']:.2f} bar"
    if BLANK:                        # TODO: is it over the token ceiling?
        return False, f"{ratio:.1f}x tokens exceeds {threshold['max_token_ratio']}x"
    return True, f"gain {gain:+.2f} at {ratio:.1f}x tokens"

In [ ]:
# --- Self-check: Section 3
_base = {"arm": "direct", "pass_rate": 0.60, "calls": 6, "tokens": 100}
_cheap_win = {"arm": "cot", "pass_rate": 0.85, "calls": 6, "tokens": 300}
_dear_win  = {"arm": "tot", "pass_rate": 0.90, "calls": 60, "tokens": 4000}
_no_gain   = {"arm": "react", "pass_rate": 0.62, "calls": 12, "tokens": 400}

check("a real gain inside the ceiling is accepted",
      lambda: verdict(_cheap_win, _base)[0] is True)
check("a big gain outside the ceiling is rejected",
      lambda: verdict(_dear_win, _base)[0] is False,
      "a 30-point gain does not license 40x the tokens -- that is what the ceiling is for")
check("a marginal gain is rejected", lambda: verdict(_no_gain, _base)[0] is False)
check("the baseline always passes against itself",
      lambda: verdict(_base, _base)[0] is True)
check("every rejection states a reason", lambda: len(verdict(_dear_win, _base)[1]) > 10)

## Section 4 &mdash; The scorecard

One table. The cheapest arm is the baseline; everything else has to earn its place against it.

In [ ]:
def scorecard() -> str:
    rows = sorted((run_arm(n, f) for n, f in ARMS.items()), key=lambda r: r["tokens"])
    baseline = rows[0]
    out = [f"{'arm':10}{'pass':>8}{'calls':>8}{'tokens':>9}{'x base':>9}  verdict",
           "-" * 78]
    for r in rows:
        ok, why = verdict(r, baseline)
        ratio = r["tokens"] / max(baseline["tokens"], 1)
        out.append(f"{r['arm']:10}{r['pass_rate']:>8}{r['calls']:>8}{r['tokens']:>9}"
                   f"{ratio:>8.1f}x  {'SHIP' if ok else 'no'} -- {why}")
    winners = [r["arm"] for r in rows if verdict(r, baseline)[0]]
    out.append("")
    out.append(f"Ships: {winners[-1]} (cheapest arm clearing the bar)")
    return "\n".join(out)

try:
    print(scorecard())
except NameError:
    print("(finish the sections above, then re-run this cell)")

In [ ]:
# --- Self-check: Section 4
def _rows():
    return sorted((run_arm(n, f) for n, f in ARMS.items()), key=lambda r: r["tokens"])

check("the table has a row per arm plus a header, a rule and a footer",
      lambda: len(scorecard().splitlines()) == len(ARMS) + 4)
check("the cheapest arm is the baseline",
      lambda: _rows()[0]["tokens"] == min(r["tokens"] for r in _rows()))
check("the tree does not ship on this eval set",
      lambda: verdict([r for r in _rows() if r["arm"] == "tot"][0], _rows()[0])[0] is False,
      "it costs an order of magnitude more for no measured gain")
check("a tie goes to the cheaper arm",
      lambda: verdict({"arm": "x", "pass_rate": _rows()[0]["pass_rate"],
                       "calls": 1, "tokens": _rows()[0]["tokens"] * 2}, _rows()[0])[0] is False)

## Run it for real

Have the model write the design-review paragraph &mdash; explaining a decision your scorecard already
made, not making one.

In [ ]:
if llm_ready():
    try:
        board = scorecard()
        summary = ask(
            "Write one short paragraph for an engineering design review. State which reasoning "
            "architecture was chosen, the evidence, and the condition under which the team should "
            "revisit it. Be plain and specific; add no claims beyond the table.\n\n"
            f"THRESHOLD AGREED IN ADVANCE: {THRESHOLD}\n\nSCORECARD:\n{board}"
        )
        print(summary)
    except NameError:
        print("(finish the sections above, then re-run this cell)")

### Read it

If the paragraph argues for the cheapest arm that cleared the bar, the process worked. If it
reaches for the tree because trees sound thorough, look at which clause let it through.

**What you take from Module 2:** a way to choose a reasoning architecture on evidence, a parser you
know the failure modes of, a failure diagnosis that picks retry or re-plan correctly, and a
scorecard that is allowed to tell you the boring answer. Module 3 gives all of it somewhere to keep
its state.

In [ ]:
score()

## Your turn

1. The arms here are deterministic, so pass rate is fixed and only cost varies. Swap `arm_cot` and
   `arm_react` for real `ask()` calls, run three times, and report the spread. How much of your
   verdict survives the variance?
2. Add a case where ReAct genuinely beats Chain-of-Thought &mdash; one whose answer is not in the prompt
   at all. That is the case that justifies tools, and your eval set currently lacks it.
3. Raise `max_token_ratio` until the tree ships. Would you defend that number to a reviewer? If not,
   you have found your real ceiling.